In [23]:
#SBERT 모델을 사용하기 위해 라이브러리 설치
# !pip install sentence-transformers

### SBERT
- BERT 모델
    - 문장 이해용 Encoder
    - 문장의 쌍을 비교
- SBERT 모델
    - 문장 의미를 임베딩
    - 벡터의 비교용

In [24]:
import torch
from sentence_transformers import SentenceTransformer, util

In [25]:
#다목적 한국 SBERT
model_name = 'jhgan/ko-sroberta-multitask'
#문장 유사도에 특화된 모델
model_name2='BM-K/KoSimCSE-roberta-multitask'

sbert=SentenceTransformer(model_name)
sbert2=SentenceTransformer(model_name2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1179.65it/s]


In [26]:
#각 모델의 설정 세팅 (최대 길이 설정)
sbert.max_seq_len=256
sbert2.max_seq_len=256

In [27]:
doc1='이 카메라는 색감이 자연스럽고 베터리도 오래 간다'
doc2='베터리 성능이 좋고 사진 품질이 뛰어나다'

In [28]:
with torch.inference_mode(): #추론 모드
    emb1=sbert.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2=sbert.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
#두 벡터 간의 유사도를 확인
cos_sim=util.cos_sim(emb1, emb2).item()
round(cos_sim,4)

0.6644

In [29]:
with torch.inference_mode(): #추론 모드
    emb1=sbert2.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2=sbert2.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
#두 벡터 간의 유사도를 확인
cos_sim=util.cos_sim(emb1, emb2).item()
round(cos_sim,4)

0.6714

In [30]:
sentences=[
    '하이닉스 주가가 올랐다',
    '코스피가 상승 마감했다',
    '날씨가 안 좋아서 항공편이 취소됬다'
]

with torch.inference_mode(): #추론 모드
    embs=sbert.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)

embs        #embs : encode() 함수의 결과 값 (CLS + 단어 벡터의 평균값)

tensor([[-0.0241, -0.0772,  0.0120,  ..., -0.0129, -0.0149, -0.0167],
        [-0.0080, -0.0036,  0.0879,  ..., -0.0145, -0.0190,  0.0652],
        [-0.0647,  0.0326,  0.1048,  ...,  0.0295, -0.0267, -0.0210]])

In [31]:
sim_metrix=util.cos_sim(embs, embs)
sim_metrix

tensor([[1.0000, 0.2660, 0.0632],
        [0.2660, 1.0000, 0.0028],
        [0.0632, 0.0028, 1.0000]])

In [32]:
new_sentence='증시가 강세였다'

new_emb=sbert2.encode(new_sentence, convert_to_tensor=True, normalize_embeddings=True)

#유사도가 높은 상위의 2개를 선택
hits=torch.topk(util.cos_sim(new_emb,embs).squeeze(0), k=2)

In [33]:
for score, idx in zip(hits.values.tolist(), hits.indices.tolist()):
    #score : 코사인 유사도 
    #idx : 위치
    print(f'유사 문장 : {sentences[idx]}, 유사도 : {round(score, 3)}')

유사 문장 : 코스피가 상승 마감했다, 유사도 : 0.384
유사 문장 : 하이닉스 주가가 올랐다, 유사도 : 0.367


#### 실습
- ratings_trai.txt 파일 로드
- 데이터 튜닝 (결측치 제거, 정규화, 중복 데이터 제거, 글자수 1자리 이하 제거)
- train, test 데이터 분할
- sbert 모델은 'jhgan/ko-sroberta-multitask' 사용
- Dataset 선언
    - 입력 받는 데이터는 텍스트, 라벨
    - 생성자 함수
        - 입력 받은 텍스트(sentences)들을 sbert 모델을 이용하여 임베딩
        - 라벨 데이터는 self 변수로 저장
    - __len__() : 길이를 되돌려준다.
    - __getitem() : 임베딩된 데이터에서 특정 위치의 데이터와 같은 위치의 라벨 데이터를 되돌려준다.
- DataLoader를 통해서 batch data 생성을 설정
- 학습 모델을 생성
    - ML
        - SVC 모델
    - DL
        - 비선형 활성화 함수를 포함한 다중 구조로 생성
- 예측값을 이용하여 평가 지표 확인

In [77]:
import re
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.svm import SVC

In [35]:
df=pd.read_csv('../data/ratings_train.txt', sep='\t')

In [36]:
def normalize(text):
    text=re.sub(r'[^가-핳0-9a-zA-Z\s\.]',' ',str(text))
    test=re.sub(r'\s+',' ',text).strip()
    return text

In [37]:
#결측치 제거
df.dropna(inplace=True)
#텍스트 정규화
df['document']=df['document'].map(normalize)
#중복 데이터 제거
df.drop_duplicates('document', inplace=True)
#길이가 1 이하인 데이터 제거
df=df.loc[df['document'].str.len()>1]

len(df)

145696

In [38]:
#랜덤한 5000개의 데이터 추출
df2=df.sample(n=5000, random_state=42).reset_index(drop=True)
df2['label'].value_counts()

label
1    2528
0    2472
Name: count, dtype: int64

In [39]:
#train,test 데이터로 분할
train_df, test_df=train_test_split(
    df2, test_size=0.2, random_state=42, stratify=df2['label']
)

In [40]:
#모델 선텍
sbert3=SentenceTransformer('jhgan/ko-sroberta-multitask')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1301.01it/s]


In [41]:
#Dataset 선언
class SBERTDataset(Dataset):
    #생성자 함수 : document, label 데이터를 입력 받는다.
    def __init__(self, document, labels):
        #생성자 함수에서 document 데이터를 임베딩
        self.labels=torch.tensor(labels, dtype=torch.long)

        #모델의 추론 모드 사용
        with torch.inference_mode():
            self.emb=sbert3.encode(
                document, convert_to_tensor=True, normalize_embeddings=True
            )
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]

In [58]:
class SBERTDataset2(Dataset):
    #생성자 함수 : document, label 데이터를 입력 받는다.
    def __init__(self, document, labels):
        #생성자 함수에서 document 데이터를 임베딩
        self.labels=torch.tensor(labels, dtype=torch.long)
        self.document=document
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        with torch.inference_mode():
            emb=sbert3.encode(
                self.document[idx], convert_to_tensor=True, normalize_embeddings=True
            )       #길이가 768인 1차원 tensor
        return emb, self.labels[idx]

In [59]:
train_ds=SBERTDataset(train_df['document'].tolist(), train_df['label'].tolist())
test_ds=SBERTDataset(test_df['document'].tolist(), test_df['label'].tolist())

In [60]:
train_ds2=SBERTDataset2(train_df['document'].tolist(), train_df['label'].tolist())
test_ds2=SBERTDataset2(test_df['document'].tolist(), test_df['label'].tolist())

In [61]:
train_dl=DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl=DataLoader(test_ds, batch_size=128, shuffle=True)

In [62]:
train_dl2=DataLoader(train_ds2, batch_size=128,shuffle=True)
test_dl2=DataLoader(test_ds2, batch_size=128,shuffle=True)

In [63]:
#다중 퍼셉트론 구조 딥러닝 모델을 생성
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_classes=2, dropout=0.5):
        super().__init__()

        #다중 퍼셉트론층 구성
        self.net=nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),              #비선형 구조를 이해하기 위한 비선형 활성화 함수
            nn.Dropout(dropout),    #과적합 방지용
            nn.Linear(hidden_dim, num_classes)
        )
    #순전파 함수 생성 - 독립 변수의 값을 받아온다.
    def forward(self,x):
        #x : 독립 변수 (SBERT 모델을 통해서 임베딩 데이터를 배치로 묶은 데이터)
        result=self.net(x)
        return result

In [64]:
#SBERT 모델의 출력 차원의 수를 확인
in_dim=sbert3.get_embedding_dimension()
in_dim

768

In [65]:
clf=MLPModel(in_dim)

criterion=nn.CrossEntropyLoss()
optimizer=optim.AdamW(clf.parameters(), lr=2e-04)

In [66]:
clf.train()

for epoch in range(1):
    total=0.0

    for X, y in train_dl2:
        #X : 임베딩 데이터들
        #y : label 데이터들

        optimizer.zero_grad()
        logits=clf(X)
        loss=criterion(logits,y)
        loss.backward
        optimizer.step()

        total += loss.item() * X.size(0)
    print(f'epoch : {epoch}, lost : {round(total / len(train_ds), 4)}')

epoch : 0, lost : 0.6944


In [ ]:
#예측
clf.eval()

y_true, y_pred=[], []

with torch.inference_mode():
    for X,y in test_dl:
        logits=clf(X)
        pred=logits.argmax(dim=-1).tolist()

        y_true += y.tolist()
        y_pred += pred

print(y_true)
print(y_pred)

In [68]:
print(classification_report(y_pred,y_true))

              precision    recall  f1-score   support

           0       1.00      0.49      0.66      1000
           1       0.00      0.00      0.00         0

    accuracy                           0.49      1000
   macro avg       0.50      0.25      0.33      1000
weighted avg       1.00      0.49      0.66      1000



c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

In [69]:
#Dataset에서 데이터를 가져온다. - 임베딩 데이터, 라벨 데이터
X_train,y_train=train_ds[0:len(train_ds)]

In [71]:
import numpy as np

In [72]:
X_train=np.array(X_train)
y_train=np.array(y_train)

C:\Users\user\AppData\Local\Temp\ipykernel_1896\1338424319.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  X_train=np.array(X_train)
C:\Users\user\AppData\Local\Temp\ipykernel_1896\1338424319.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_train=np.array(y_train)


In [74]:
X_test,y_test=test_ds[0:len(test_ds)]
X_test=np.array(X_test)
y_test=np.array(y_test)

C:\Users\user\AppData\Local\Temp\ipykernel_1896\1032523156.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  X_test=np.array(X_test)
C:\Users\user\AppData\Local\Temp\ipykernel_1896\1032523156.py:3: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_test=np.array(y_test)


In [78]:
#SVC모델을 생성
svc=SVC(random_state=42)

In [79]:
svc.fit(X_train,y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None


In [80]:
#예측값 생성
pred=svc.predict(X_test)

print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       0.84      0.86      0.85       494
           1       0.86      0.84      0.85       506

    accuracy                           0.85      1000
   macro avg       0.85      0.85      0.85      1000
weighted avg       0.85      0.85      0.85      1000

